# 과제 — 보안 공지 요약 봇

영문 보안 공지 10건을 모델 4개에 똑같이 넣고, **네 줄 요약**을 얼마나 정확하게 하는지 비교합니다.

| | 모델 | 돈 | 공지문이 나가는 곳 |
|---|---|---|---|
| 클라우드 | `gpt-4.1-mini`, `gpt-4.1-nano` | 토큰당 과금 | 외부 API 서버 |
| 로컬 | `qwen2.5:3b`, `llama3.2` | 0원 | 내 PC 밖으로 안 나감 |

> **질문: 공짜이고 밖으로 나가지 않는 대신, 무엇을 얼마나 잃는가?**

## ① 준비 — 라이브러리, 클라이언트, 데이터

In [1]:
import json
import os
import re
import time
from datetime import datetime

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

# 로컬 모델은 OpenAI 호환 주소(/v1)로 부릅니다. .env 에 /v1 이 없으면 붙여 줍니다.
ollama_url = os.getenv("OLLAMA_URL", "http://localhost:11434").rstrip("/")
if not ollama_url.endswith("/v1"):
    ollama_url += "/v1"

openai_client = OpenAI()                                       # 클라우드
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)  # 로컬 (키는 아무 값)

print("OpenAI key :", "있음" if os.getenv("OPENAI_API_KEY") else "없음 — .env 확인")
print("Ollama URL :", ollama_url)

# 오래 걸리는 40회 호출 전에, 로컬 모델이 정말 떠 있는지 먼저 확인합니다.
try:
    installed = sorted(m.id for m in ollama_client.models.list())
    print("로컬 모델  :", ", ".join(installed))
except Exception as e:
    print("로컬 모델  : 연결 실패 —", type(e).__name__, "(터미널에서 ollama list 확인)")

with open("advisories_sample.json", encoding="utf-8") as f:
    advisories = json.load(f)

print(f"\n공지 {len(advisories)}건")
for a in advisories:
    print(f"{a['no']:2}. {a['cve_id']:16} {a['label']:22} {len(a['advisory']):5}자")

OpenAI key : 있음
Ollama URL : http://localhost:11434/v1


로컬 모델  : llama3.2:3b, llama3.2:latest, qwen2.5:3b

공지 10건
 1. CVE-2021-44228   Apache Log4j2           2500자
 2. CVE-2022-22965   Spring Framework        1830자
 3. CVE-2017-5638    Apache Struts 2         1726자
 4. CVE-2014-6271    GNU Bash                1011자
 5. CVE-2023-49105   ownCloud                 383자
 6. CVE-2026-48710   Starlette               2500자
 7. CVE-2026-59822   LiteLLM                 1752자
 8. CVE-2026-75650   Adobe Commerce           350자
 9. CVE-2026-49869   Kestra                   772자
10. CVE-2026-73570   Zimbra Collaboration     418자


## ② 요약 프롬프트 — 네 줄로 고정 (필수 1)

- 형식을 좁혀야 **채점**할 수 있습니다.
- `"공지에 없는 내용은 지어내지 말 것"`과, 모를 때 쓸 말 `"공지에 없음"`을 **미리 정해 줍니다.**
- `label`(정답 제품명)은 모델에게 주지 않습니다. `advisory` 본문만 넣습니다.

In [2]:
NO_VERSION = "공지에 없음"

SYSTEM_PROMPT = f"""당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 정확히 네 줄만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "{NO_VERSION}">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 없는 내용은 지어내지 마세요. 공지에 없는 버전·날짜·수치를 만들지 마세요.
- 고쳐진 버전이 공지에 없으면 반드시 "{NO_VERSION}"이라고 쓰세요. 추측하지 마세요.
- 각 줄은 "제품:", "영향:", "조치:", "긴급도:"로 시작하세요. 앞에 기호(#, -, *)를 붙이지 마세요.
- 한국어로 답하세요."""


def build_messages(advisory_text, system_prompt=SYSTEM_PROMPT):
    """공지 본문 하나를 messages 로 만든다. (label 은 넣지 않는다)"""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"다음 보안 공지를 요약하세요.\n\n{advisory_text}"},
    ]


print(SYSTEM_PROMPT)

당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 정확히 네 줄만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 없는 내용은 지어내지 마세요. 공지에 없는 버전·날짜·수치를 만들지 마세요.
- 고쳐진 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.
- 각 줄은 "제품:", "영향:", "조치:", "긴급도:"로 시작하세요. 앞에 기호(#, -, *)를 붙이지 마세요.
- 한국어로 답하세요.


## ③ 모델 딕셔너리 — 모델은 코드가 아니라 데이터 (필수 2)

- 모델을 추가할 때 **이 딕셔너리에 한 줄만** 넣으면 됩니다.
- 로컬 모델도 같은 `OpenAI` 클래스입니다. 다른 것은 `base_url`(클라이언트) 하나뿐입니다.
- 세 번째 값은 `max_tokens` 입니다. 잘림이 나오면 **그 모델만** 올립니다 (⑧).

In [3]:
# 이름: (클라이언트, 모델 ID, max_tokens)
MODELS = {
    "gpt-4.1-mini": (openai_client, "gpt-4.1-mini", 300),
    "gpt-4.1-nano": (openai_client, "gpt-4.1-nano", 300),
    "qwen2.5:3b":   (ollama_client, "qwen2.5:3b",   300),
    "llama3.2":     (ollama_client, "llama3.2",     300),
}

for name, (client, model_id, mt) in MODELS.items():
    where = "로컬" if client is ollama_client else "클라우드"
    print(f"{name:14} {where:4}  model={model_id:14} max_tokens={mt}")

gpt-4.1-mini   클라우드  model=gpt-4.1-mini   max_tokens=300
gpt-4.1-nano   클라우드  model=gpt-4.1-nano   max_tokens=300
qwen2.5:3b     로컬    model=qwen2.5:3b     max_tokens=300
llama3.2       로컬    model=llama3.2       max_tokens=300


## ④ 호출 함수 — 클라우드인지 로컬인지 몰라도 된다

- `temperature=0`: 같은 입력에 같은 답이 나와야 비교가 됩니다.
- 답변과 함께 `finish_reason`, `usage`(실제 토큰), 걸린 시간을 기록합니다.
- 이 함수는 예외를 그대로 올리고, 실행 루프(⑦)에서 `try/except`로 잡습니다.

In [4]:
def ask(client, model_id, advisory_text, max_tokens, system_prompt=SYSTEM_PROMPT, temperature=0):
    """공지 하나를 요약시킨다. 결과와 메타데이터를 함께 돌려준다."""
    started = time.time()
    response = client.chat.completions.create(
        model=model_id,
        messages=build_messages(advisory_text, system_prompt),
        temperature=temperature,
        max_tokens=max_tokens,
    )
    choice = response.choices[0]
    usage = response.usage
    return {
        "answer": choice.message.content or "",
        "finish_reason": choice.finish_reason,
        "in_tokens": usage.prompt_tokens if usage else 0,
        "out_tokens": usage.completion_tokens if usage else 0,
        "seconds": round(time.time() - started, 1),
        "max_tokens": max_tokens,
        "error": None,
    }

## ⑤ 보내기 전에 토큰 세기 (필수 3)

`tiktoken`은 API를 부르지 않고 내 PC에서 셉니다. **돈을 쓰기 전에** 크기를 가늠할 수 있습니다.
(Qwen·Llama는 토크나이저가 달라 정확히 일치하지 않습니다. 실제 값은 응답의 `usage`로 따로 받습니다.)

In [5]:
encoding = tiktoken.get_encoding("o200k_base")   # gpt-4.1 계열 토크나이저


def count_tokens(messages):
    return sum(len(encoding.encode(m["content"])) for m in messages)


pre_tokens = {a["no"]: count_tokens(build_messages(a["advisory"])) for a in advisories}

for a in advisories:
    print(f"{a['no']:2}. {a['cve_id']:16} {pre_tokens[a['no']]:>5} 토큰")
total_pre = sum(pre_tokens.values())
print(f"\n10건 합계 {total_pre:,} 토큰 (모델 1개 기준), 모델 {len(MODELS)}개면 약 {total_pre * len(MODELS):,} 토큰")

 1. CVE-2021-44228     828 토큰
 2. CVE-2022-22965     672 토큰
 3. CVE-2017-5638      612 토큰
 4. CVE-2014-6271      409 토큰
 5. CVE-2023-49105     313 토큰
 6. CVE-2026-48710     711 토큰
 7. CVE-2026-59822     599 토큰
 8. CVE-2026-75650     289 토큰
 9. CVE-2026-49869     389 토큰
10. CVE-2026-73570     308 토큰

10건 합계 5,130 토큰 (모델 1개 기준), 모델 4개면 약 20,520 토큰


## ⑥ 채점표와 채점 함수 (필수 5)

채점표는 `exercise.md` 4장을 그대로 옮겼습니다.

- **형식**: 네 줄이 **줄 머리부터** `제품:` `영향:` `조치:` `긴급도:`로 시작하는가
- **제품**: 정답 제품 낱말이 답에 있는가
- **조치**: 정답 버전 낱말이 `조치:` 줄에 있는가. 4번·8번은 `공지에 없음`이 정답
- **지어냄**: 4번·8번의 `조치:` 줄에 `숫자.숫자` 꼴의 버전이 있으면 지어낸 것

> 형식 검사와 줄 꺼내기(`get_line`)가 **같은 기준**(`strip().startswith`)을 써야 합니다.
> `"제품:" in answer`로 검사하면 `# 제품:`도 통과하는데, 조치 줄은 못 꺼내서 점수가 거짓말을 합니다.

In [6]:
# no: (제품 낱말, [조치 낱말 — 하나만 맞으면 정답])  빈 목록 = "공지에 없음"이 정답
ANSWER_KEY = {
    1:  ("Log4j",          ["2.16.0"]),
    2:  ("Spring",         ["5.3.18", "5.2.20"]),
    3:  ("Struts",         ["2.3.32", "2.5.10.1"]),
    4:  ("Bash",           []),
    5:  ("ownCloud",       ["10.13.1"]),
    6:  ("Starlette",      ["1.0.1"]),
    7:  ("LiteLLM",        ["1.84.0"]),
    8:  ("Adobe Commerce", []),
    9:  ("Kestra",         ["1.0.45", "1.3.21"]),
    10: ("Zimbra",         ["10.1.20"]),
}
REQUIRED_LINES = ["제품:", "영향:", "조치:", "긴급도:"]


def has_line(answer, prefix):
    return any(line.strip().startswith(prefix) for line in answer.splitlines())


def get_line(answer, prefix):
    for line in answer.splitlines():
        if line.strip().startswith(prefix):
            return line.split(":", 1)[1].strip()
    return ""


def grade(no, answer):
    product, versions = ANSWER_KEY[no]
    form = all(has_line(answer, p) for p in REQUIRED_LINES)
    prod = product.lower() in answer.lower()
    action = get_line(answer, "조치:")
    if versions:
        act = any(v in action for v in versions)
        made_up = False
    else:
        # 공백을 양쪽 다 지우고 비교 ("공지에없음"처럼 붙여 쓰는 모델 대비)
        act = NO_VERSION.replace(" ", "") in action.replace(" ", "")
        made_up = bool(re.search(r"\d+\.\d+", action))
    return {"형식": form, "제품": prod, "조치": act, "지어냄": made_up}


# 채점 함수 자체를 먼저 시험합니다. 채점 버그는 에러 없이 조용히 틀린 점수를 줍니다.
assert grade(1, "제품: Log4j\n영향: x\n조치: 2.16.0 이상\n긴급도: 긴급") == {"형식": True, "제품": True, "조치": True, "지어냄": False}
assert grade(1, "# 제품: Log4j\n# 영향: x\n# 조치: 2.16.0\n# 긴급도: 긴급")["형식"] is False
assert grade(4, "제품: Bash\n영향: x\n조치: 공지에없음\n긴급도: 높음")["조치"] is True
assert grade(4, "제품: Bash\n영향: x\n조치: 4.4 이상\n긴급도: 높음")["지어냄"] is True
print("채점 함수 시험 통과 — 버전 없는 공지:", [n for n, (_, v) in ANSWER_KEY.items() if not v])

채점 함수 시험 통과 — 버전 없는 공지: [4, 8]


## ⑦ 전체 실행 — 4개 모델 × 10건 = 40회 (필수 2, 4)

- 호출마다 `try/except` — 한 모델(또는 한 건)이 실패해도 **나머지는 끝까지** 돕니다.
- 출력의 `OOO`는 왼쪽부터 **형식·제품·조치**입니다.
- `finish_reason`이 `length`면 잘린 답입니다.

In [7]:
def run_model(name, max_tokens=None, only=None):
    """모델 하나로 공지들을 돌린다. only 에 번호 목록을 주면 그 건만 돈다."""
    client, model_id, default_mt = MODELS[name]
    mt = max_tokens or default_mt
    out = {}
    for a in advisories:
        if only and a["no"] not in only:
            continue
        try:
            r = ask(client, model_id, a["advisory"], mt)
        except Exception as e:
            r = {"answer": "", "finish_reason": None, "in_tokens": 0, "out_tokens": 0,
                 "seconds": 0.0, "max_tokens": mt, "error": f"{type(e).__name__}: {e}"}
            print(f"  {a['no']:2}. {a['cve_id']:16} [오류] {r['error'][:70]}")
        else:
            g = grade(a["no"], r["answer"])
            mark = "".join("O" if g[k] else "X" for k in ("형식", "제품", "조치"))
            warn = "  <- 지어냄" if g["지어냄"] else ""
            print(f"  {a['no']:2}. {a['cve_id']:16} {mark}  {r['finish_reason']:6} {r['seconds']:5.1f}초{warn}")
        out[a["no"]] = r
    return out


run_started = datetime.now()
results = {}
for name in MODELS:
    print(f"===== {name} =====")
    results[name] = run_model(name)
    print()
print("완료:", datetime.now().strftime("%H:%M:%S"))

===== gpt-4.1-mini =====


   1. CVE-2021-44228   OOO  stop     2.2초


   2. CVE-2022-22965   OOO  stop     1.3초


   3. CVE-2017-5638    OOO  stop     1.2초


   4. CVE-2014-6271    OOO  stop     1.3초


...

   6. CVE-2026-48710   OOO  stop    21.2초


   7. CVE-2026-59822   OOO  stop    22.2초


   8. CVE-2026-75650   OOX  stop     7.8초  <- 지어냄


   9. CVE-2026-49869   OOO  stop     9.2초


  10. CVE-2026-73570   OOX  stop     9.0초

완료: 10:35:08


## ⑧ 잘림 검사와 재실행 (필수 4)

`finish_reason == "length"`인 건을 셉니다. 잘린 답으로 매긴 점수는 모델 실력이 아니라 **내 설정**을 잰 것입니다.
잘린 모델이 있으면 **그 모델만** `max_tokens`를 올려 잘린 건을 다시 돌리고, 결과를 바꿔 끼웁니다.

In [8]:
RETRY_MAX_TOKENS = 1024
truncation_log = []

for name in MODELS:
    cut = [no for no, r in results[name].items() if r["finish_reason"] == "length"]
    print(f"{name:14} 잘림 {len(cut)}건 {cut if cut else ''}")
    if cut:
        print(f"  -> max_tokens {MODELS[name][2]} -> {RETRY_MAX_TOKENS} 로 다시 실행")
        retry = run_model(name, max_tokens=RETRY_MAX_TOKENS, only=cut)
        still = [no for no, r in retry.items() if r["finish_reason"] == "length"]
        truncation_log.append((name, cut, still))
        results[name].update(retry)
        print(f"  -> 다시 돌린 뒤에도 잘림 {len(still)}건 {still if still else ''}")

if not truncation_log:
    print("\n잘린 답 없음 — max_tokens=300 이 네 모델 모두에게 충분했습니다.")

gpt-4.1-mini   잘림 0건 
gpt-4.1-nano   잘림 0건 
qwen2.5:3b     잘림 0건 
llama3.2       잘림 0건 

잘린 답 없음 — max_tokens=300 이 네 모델 모두에게 충분했습니다.


## ⑨ 점수표 (필수 3, 5)

`지어냄`은 더하는 점수가 아니라 **거르는 조건**입니다. 그리고 `지어냄 0`만 보면 안 됩니다 —
전부 "공지에 없음"이라고만 답해도, 호출이 전부 실패해도 0입니다. **조치 점수와 오류 건수를 같이** 봅니다.

In [9]:
def summarize(name):
    s = {"형식": 0, "제품": 0, "조치": 0, "지어냄": 0, "잘림": 0, "오류": 0,
         "입력토큰": 0, "출력토큰": 0, "총초": 0.0}
    for a in advisories:
        r = results[name][a["no"]]
        if r["error"]:
            s["오류"] += 1
            continue
        g = grade(a["no"], r["answer"])
        for k in ("형식", "제품", "조치", "지어냄"):
            s[k] += int(g[k])
        s["잘림"] += int(r["finish_reason"] == "length")
        s["입력토큰"] += r["in_tokens"]
        s["출력토큰"] += r["out_tokens"]
        s["총초"] += r["seconds"]
    s["총토큰"] = s["입력토큰"] + s["출력토큰"]
    s["총초"] = round(s["총초"], 1)
    s["건당초"] = round(s["총초"] / max(1, len(advisories) - s["오류"]), 1)
    return s


scores = {name: summarize(name) for name in MODELS}
table = pd.DataFrame(scores).T[["형식", "제품", "조치", "지어냄", "잘림", "오류",
                                "입력토큰", "출력토큰", "총토큰", "총초", "건당초"]]
n = len(advisories)
display_table = table.copy()
for k in ("형식", "제품", "조치"):
    display_table[k] = table[k].astype(int).astype(str) + f"/{n}"
display_table

,형식,제품,조치,지어냄,잘림,오류,입력토큰,출력토큰,총토큰,총초,건당초
gpt-4.1-mini,10/10,10/10,10/10,0.0,0.0,0.0,5240.0,659.0,5899.0,14.5,1.4
gpt-4.1-nano,10/10,10/10,6/10,0.0,0.0,0.0,5240.0,567.0,5807.0,9.6,1.0
qwen2.5:3b,10/10,10/10,8/10,0.0,0.0,0.0,5498.0,911.0,6409.0,158.6,15.9
llama3.2,10/10,10/10,6/10,2.0,0.0,0.0,5405.0,756.0,6161.0,164.8,16.5


### (선택) 금액 환산

단가는 자주 바뀝니다. 아래는 **OpenAI 공개 단가(1M 토큰당 USD)를 가정한 값**이니, 쓰기 전에 가격 페이지에서 확인하세요.
로컬 모델은 0원이지만 대신 **시간**(총초)을 씁니다.

In [10]:
PRICE_PER_1M = {   # (입력, 출력) USD — 가정값, https://openai.com/api/pricing 에서 확인
    "gpt-4.1-mini": (0.40, 1.60),
    "gpt-4.1-nano": (0.10, 0.40),
}
for name, s in scores.items():
    if name in PRICE_PER_1M:
        pin, pout = PRICE_PER_1M[name]
        usd = s["입력토큰"] / 1e6 * pin + s["출력토큰"] / 1e6 * pout
        print(f"{name:14} 10건 ${usd:.5f}  -> 공지 1만 건이면 약 ${usd * 1000:,.2f}")
    else:
        print(f"{name:14} 0원, 대신 {s['총초']}초 (건당 {s['건당초']}초) -> 1만 건이면 약 {s['건당초'] * 10000 / 3600:,.1f}시간")

gpt-4.1-mini   10건 $0.00315  -> 공지 1만 건이면 약 $3.15
gpt-4.1-nano   10건 $0.00075  -> 공지 1만 건이면 약 $0.75
qwen2.5:3b     0원, 대신 158.6초 (건당 15.9초) -> 1만 건이면 약 44.2시간
llama3.2       0원, 대신 164.8초 (건당 16.5초) -> 1만 건이면 약 45.8시간


## ⑩ 4번·8번 — 지어냈는가 (3장)

두 공지에는 고쳐진 버전이 **없습니다.** 정답은 `조치: 공지에 없음`입니다.
그럴듯한 가짜 버전은 **틀린 답보다 나쁩니다** — 사용자가 걸러낼 방법이 없기 때문입니다.

In [11]:
for no in (4, 8):
    label = next(a["label"] for a in advisories if a["no"] == no)
    print("=" * 72)
    print(f"{no}번 — {label}  (공지에 패치 버전 없음)")
    print("=" * 72)
    for name in MODELS:
        r = results[name][no]
        if r["error"]:
            print(f"  {name:14} [오류]")
            continue
        g = grade(no, r["answer"])
        verdict = "지어냄 !!" if g["지어냄"] else ("정답" if g["조치"] else "애매")
        print(f"  {name:14} [{verdict:6}] 조치: {get_line(r['answer'], '조치:')[:70]}")
    print()

4번 — GNU Bash  (공지에 패치 버전 없음)
  gpt-4.1-mini   [정답    ] 조치: 공지에 없음
  gpt-4.1-nano   [정답    ] 조치: 공지에 없음
  qwen2.5:3b     [정답    ] 조치: 공지에 없음
  llama3.2       [지어냄 !!] 조치: GNU Bash 4.4 이상

8번 — Adobe Commerce  (공지에 패치 버전 없음)
  gpt-4.1-mini   [정답    ] 조치: 공지에 없음
  gpt-4.1-nano   [정답    ] 조치: 공지에 없음
  qwen2.5:3b     [정답    ] 조치: 공지에 없음
  llama3.2       [지어냄 !!] 조치: Adobe Commerce 2.3.3f1 및 이후 버전을 올려야 합니다.



### 틀린 답 열어 보기

점수표만 보면 "몇 점짜리 모델"로만 보입니다. 틀린 답을 열어 봐야 **왜** 틀렸는지 압니다.

In [12]:
for name in MODELS:
    wrong = [a for a in advisories
             if not results[name][a["no"]]["error"]
             and not all(grade(a["no"], results[name][a["no"]]["answer"])[k] for k in ("형식", "제품", "조치"))]
    print(f"===== {name}: 틀린 건 {len(wrong)}개 =====")
    for a in wrong:
        g = grade(a["no"], results[name][a["no"]]["answer"])
        miss = [k for k in ("형식", "제품", "조치") if not g[k]]
        print(f"--- {a['no']}. {a['label']}  (틀린 항목: {', '.join(miss)}, 정답 조치: {ANSWER_KEY[a['no']][1] or NO_VERSION})")
        print(results[name][a["no"]]["answer"].strip())
    print()

===== gpt-4.1-mini: 틀린 건 0개 =====

===== gpt-4.1-nano: 틀린 건 4개 =====
--- 1. Apache Log4j2  (틀린 항목: 조치, 정답 조치: ['2.16.0'])
제품: Log4j 2.14.1 이하
영향: 원격 코드 실행(RCE) 공격이 가능
조치: 공지에 없음
긴급도: 높음
--- 3. Apache Struts 2  (틀린 항목: 조치, 정답 조치: ['2.3.32', '2.5.10.1'])
제품: Apache Struts 2 2.3.x 이전 버전, 2.5.x 이전 버전
영향: 원격 공격자가 조작된 HTTP 헤더를 통해 임의의 명령을 실행할 수 있음
조치: 공지에 없음
긴급도: 높음
--- 5. ownCloud  (틀린 항목: 조치, 정답 조치: ['10.13.1'])
제품: ownCloud core 버전 10.6.0 이전
영향: 공격자가 인증 없이 파일에 접근, 수정 또는 삭제 가능
조치: 공지에 없음
긴급도: 높음
--- 10. Zimbra Collaboration  (틀린 항목: 조치, 정답 조치: ['10.1.20'])
제품: Zimbra Collaboration (ZCS) 버전 10.1.20 미만  
영향: 인증되지 않은 공격자가 임의의 운영체제 명령을 실행할 수 있음  
조치: 공지에 없음  
긴급도: 높음

===== qwen2.5:3b: 틀린 건 2개 =====
--- 3. Apache Struts 2  (틀린 항목: 조치, 정답 조치: ['2.3.32', '2.5.10.1'])
제품: Apache Struts 2 2.3.x, 2.5.x
영향: 공격자가 malicious Content-Type 값을 사용하여 원격 코드 실행을 수행할 수 있습니다.
조치: 공지에 없음
긴급도: 높음
--- 10. Zimbra Collaboration  (틀린 항목: 조치, 정답 조치: ['10.1.20'])
제품: Zimbra Collaboration (ZCS) before 10.1.20
영향: 악의적인 사용

### "공지에 없음" 지시 한 줄 — 빼고 재 보기 (필수 1)

`system`에서 **모를 때 쓸 말** 한 줄을 빼면 어떻게 되는지 직접 잽니다.
한 번만 재고 결론 내지 않도록, 4번·8번을 **세 번씩** 돌립니다. (`gpt-4.1-mini` 기준, 12회 호출)

In [13]:
SYSTEM_NO_GUARD = """당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 정확히 네 줄만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전>
긴급도: <긴급|높음|보통|낮음 중 하나>

한국어로 답하세요."""

guard_rows = []
for no in (4, 8):
    a = next(x for x in advisories if x["no"] == no)
    print("=" * 72)
    print(f"{no}번 — {a['label']}")
    for tag, sp in [("지시 있음", SYSTEM_PROMPT), ("지시 없음", SYSTEM_NO_GUARD)]:
        for trial in range(1, 4):
            try:
                ans = ask(openai_client, "gpt-4.1-mini", a["advisory"], 300, system_prompt=sp)["answer"]
            except Exception as e:
                print(f"  [{tag}] {trial}회  [오류] {type(e).__name__}")
                continue
            action = get_line(ans, "조치:")
            made_up = bool(re.search(r"\d+\.\d+", action))
            guard_rows.append((no, tag, trial, made_up, action))
            print(f"  [{tag}] {trial}회  {'지어냄 !!' if made_up else '안 지어냄'}  조치: {action[:60]}")
    print()

4번 — GNU Bash


  [지시 있음] 1회  안 지어냄  조치: 공지에 없음


  [지시 있음] 2회  안 지어냄  조치: 공지에 없음


  [지시 있음] 3회  안 지어냄  조치: 공지에 없음


  [지시 없음] 1회  지어냄 !!  조치: 4.3 이후 버전으로 업데이트


...

  [지시 있음] 2회  안 지어냄  조치: 공지에 없음


  [지시 있음] 3회  안 지어냄  조치: 공지에 없음


  [지시 없음] 1회  안 지어냄  조치: 최신 버전으로 업데이트 권장


  [지시 없음] 2회  안 지어냄  조치: 최신 보안 패치 적용 권고


  [지시 없음] 3회  안 지어냄  조치: 최신 버전으로 업데이트 권장



## ⑪ 결과 저장 — `exercise_result.md` (필수 6)

실행 시각, 점수표, 4번·8번 비교, 모델별 전체 답변을 저장합니다. `encoding="utf-8"` 필수.

In [14]:
def save_markdown(filepath="exercise_result.md"):
    for need in ("results", "scores"):
        if need not in globals():
            raise RuntimeError(f"{need} 가 없습니다. ⑦·⑨ 셀을 먼저 실행하세요.")
    n = len(advisories)
    lines = [
        "# 보안 공지 요약 — 모델 비교 결과",
        "",
        f"- 실행 시각: {run_started.strftime('%Y-%m-%d %H:%M:%S')}",
        f"- 공지 {n}건 × 모델 {len(MODELS)}개 = {n * len(MODELS)}회 호출, temperature=0",
        "",
        "## 점수표",
        "",
        "| 모델 | 형식 | 제품 | 조치 | 지어냄 | 잘림 | 오류 | 입력토큰 | 출력토큰 | 총토큰 | 총 초 | 건당 초 |",
        "|---|---|---|---|---|---|---|---|---|---|---|---|",
    ]
    for name, s in scores.items():
        lines.append(f"| {name} | {s['형식']}/{n} | {s['제품']}/{n} | {s['조치']}/{n} | **{s['지어냄']}** | "
                     f"{s['잘림']} | {s['오류']} | {s['입력토큰']:,} | {s['출력토큰']:,} | {s['총토큰']:,} | "
                     f"{s['총초']} | {s['건당초']} |")
    if truncation_log:
        lines += ["", "## 잘림 재실행", ""]
        for name, cut, still in truncation_log:
            lines.append(f"- {name}: 잘림 {cut} → max_tokens {RETRY_MAX_TOKENS}로 재실행 후 남은 잘림 {still}")
    lines += ["", "## 버전이 공지에 없는 두 건 (4번·8번)", ""]
    for no in (4, 8):
        label = next(a["label"] for a in advisories if a["no"] == no)
        lines += [f"### {no}번 — {label}", ""]
        for name in MODELS:
            r = results[name][no]
            action = "[오류]" if r["error"] else get_line(r["answer"], "조치:")
            flag = " ← 지어냄" if not r["error"] and grade(no, r["answer"])["지어냄"] else ""
            lines.append(f"- **{name}**: {action}{flag}")
        lines.append("")
    if guard_rows:
        lines += ["## '공지에 없음' 지시를 뺐을 때 (gpt-4.1-mini, 3회씩)", "",
                  "| 번호 | 지시 | 회차 | 지어냄 | 조치 |", "|---|---|---|---|---|"]
        for no, tag, trial, made_up, action in guard_rows:
            lines.append(f"| {no} | {tag} | {trial} | {'예' if made_up else '아니오'} | {action} |")
        lines.append("")
    lines += ["## 모델별 전체 답변", ""]
    for name in MODELS:
        lines += [f"### {name}", ""]
        for a in advisories:
            r = results[name][a["no"]]
            lines += [f"**{a['no']}. {a['cve_id']} ({a['label']})** — finish_reason={r['finish_reason']}, "
                      f"{r['in_tokens']}+{r['out_tokens']} 토큰, {r['seconds']}초", "",
                      "```", r["error"] or r["answer"].strip(), "```", ""]
    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"저장: {filepath} ({len(lines)}줄)")


save_markdown()

저장: exercise_result.md (297줄)


## ⑫ 결론

실행 결과(2026-10-13, temperature=0, 4개 모델 × 10건)를 근거로 답합니다.

| 모델 | 조치 | 지어냄 | 총 토큰 | 건당 시간 | 10건 비용(가정 단가) |
|---|---|---|---|---|---|
| gpt-4.1-mini | **10/10** | 0 | 5,899 | 1.4초 | 약 $0.003 |
| gpt-4.1-nano | 6/10 | 0 | 5,807 | 1.0초 | 약 $0.0008 |
| qwen2.5:3b | 8/10 | 0 | 6,409 | 15.9초 | 0원 |
| llama3.2 | 6/10 | **2** | 6,161 | 16.5초 | 0원 |

(형식·제품은 네 모델 모두 10/10이라 모델을 가르지 못했습니다. 잘림·오류도 모두 0건입니다.)

**1. 어느 모델을 고르겠는가?**
조건이 없다면 `gpt-4.1-mini`를 고릅니다. 조치 10/10으로 유일하게 만점이고, 지어냄 0, 잘림 0입니다.
10건에 약 $0.003(가정 단가 기준), 공지 1만 건이라도 약 $3 수준이라 비용 부담도 작습니다.

**2. 4번·8번에서 지어낸 모델이 있었는가?**
`llama3.2`가 두 건 모두 지어냈습니다. 4번(Bash)에는 공지에 없는 `4.4 이상`을, 8번(Adobe Commerce)에는 공지에 숫자가 하나도 없는데 `2.3.3f1 및 이후 버전`을 만들어 냈습니다.
나머지 세 모델은 두 건 모두 `공지에 없음`으로 정확히 답했습니다.
또 `gpt-4.1-mini`에서 "공지에 없음" 지시 한 줄을 빼 보니, 4번에서 3회 중 3회 모두 `4.3 이후 버전으로 업데이트`처럼 공지의 `through 4.3`(4.3까지 **취약**)을 고쳐진 버전처럼 바꿔 썼습니다. 모를 때 쓸 말을 정해 주는 한 줄이 지어냄을 막고 있었습니다.

**3. 점수가 같다면 무엇으로 고르겠는가?**
`gpt-4.1-nano`와 `llama3.2`는 조치 6/10으로 같지만, llama3.2는 지어냄이 2건이라 보안 업무에서는 탈락입니다(지어냄은 점수가 아니라 거르는 조건).
지어냄까지 같다면 클라우드에서는 토큰(=돈), 로컬에서는 시간으로 고릅니다.
다만 nano는 mini보다 토큰이 1.5%밖에 적지 않으면서 조치가 4건 적었으므로, 이 일에서는 싼 모델을 고를 이유가 약합니다.

**4. 공지를 외부로 보낼 수 없는 조직이라면?**
`qwen2.5:3b`를 쓰겠습니다. 같은 로컬 3B급인 llama3.2보다 조치가 2건 더 맞고(8/10 vs 6/10) 지어냄이 0입니다.
감수할 손해는 숫자로 이렇습니다.
- 정확도: 조치 10/10 → 8/10, **2건(20%p) 손실**. 틀린 두 건(3번 Struts, 10번 Zimbra)은 버전을 `제품:` 줄에 써 놓고 `조치:`에는 옮기지 못한 경우라, 사람이 답을 한 번 더 확인해야 합니다.
- 시간: 건당 1.4초 → 15.9초, **약 11배 느림**. 공지 1만 건이면 약 44시간입니다(이 PC는 GPU 없이 CPU로 실행, 시간은 실행마다 흔들리므로 범위로 봐야 함).
- 대신 비용은 0원이고, 공지문이 PC 밖으로 나가지 않습니다.
